# Persona steering (EXPERIMENTAL)

These persona variants are experimental. In the j-steer-dev experiments,
persona-contrast pullbacks FAILED specificity controls: they steered
generations, but no more selectively than an unrelated persona's vector did.
(A "pullback" here is `J_l^T @ w`: a direction `w` at the output pulled back to a
residual-stream direction, the standard autodiff name for J-transpose applied to
a cotangent.) Only `word_vector` (see `word_steering.ipynb`) is the verified
method. This notebook exists so you can experiment and compare against a plain
mean_diff baseline, not as a recommendation.

Two variants:

- `persona_vector`: pull back the final-layer activation contrast
  `h_bar(pos) - h_bar(neg)` through the cached Jacobian.
- `persona_topk_vector`: read each persona's mean activation through the
  unembedding, take the top-k tokens it evokes, contrast those tokens'
  unembedding rows, pull that back (persona -> vocabulary bottleneck -> the
  verified word mechanism).

In [ ]:
# demo notebook authored by Claude
import sys
from loguru import logger

logger.remove()  # show_steer prints through loguru; route it to the cell output
logger.add(sys.stdout, format="{message}")

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from jsteer import Jacobian, show_steer

sys.path.insert(0, "..")  # repo root for config.py
import config

MODEL = "Qwen/Qwen3.5-4B"  # 4B-class: demo material. 0.6B degenerates too easily.
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16).to("cuda").eval()

# fit-or-load the cache for THIS model (chat-templated WikiText; fit where we steer).
# The lambda means WikiText is only built on a cache MISS. dim_batch=4 fits 4B on a 3090.
jac = Jacobian.fit_cached(model, tok, lambda: config.chat_corpus(tok, 128),
                          config.cache_path(MODEL), layers=(0.3, 0.9), dim_batch=4)
jac

## The persona contrast: optimist vs pessimist

Eight short first-person statements per side. These are the prompts whose
final-layer mean activations get contrasted (and, for the mean_diff baseline,
the pos/neg training prompts).

In [ ]:
optimist = [
    "Things usually work out better than people expect, and today is no exception.",
    "Every setback I have hit this year turned into a door I could not have planned for.",
    "The team is behind schedule, but honestly the hard part is done and the rest is downhill.",
    "I love how much there is to look forward to this month.",
    "Even the rainy days lately have felt like a good excuse to slow down and enjoy the quiet.",
    "The new neighbours seem wonderful, and I think this street keeps getting friendlier.",
    "Whatever happens with the results, we learned so much that we already came out ahead.",
    "I woke up early, the coffee was perfect, and I am certain this week is going to be great.",
]
pessimist = [
    "Things usually go worse than people expect, and today is no exception.",
    "Every setback this year just confirmed that planning is pointless.",
    "The team is behind schedule, and frankly the hardest part has not even started.",
    "I dread how much is crammed into this month.",
    "The rainy days lately just make everything feel heavier and more pointless.",
    "The new neighbours seem like trouble, and this street keeps getting worse.",
    "Whatever happens with the results, it will not make up for the time we wasted.",
    "I woke up tired, the coffee was burnt, and I am certain this week is going to drag.",
]

DEMO = "Give me your honest assessment of how the project is going."

## persona_vector (EXPERIMENTAL)

Pulls `h_bar(optimist) - h_bar(pessimist)` back through the Jacobian.
SHOULD: +C reads more upbeat than C=0, but expect the effect blunter and less
specific than the word vector, and possibly degenerate at large |C| (this is the
method that failed specificity controls in j-steer-dev). Watch the j-space row
and the `<think>` trace to judge whether the tone moved coherently or just broke.

In [ ]:
v_persona = jac.persona_vector(model, tok, optimist, pessimist)
show_steer(jac, model, tok, v_persona, DEMO, Cs=(-4, 0, 4))

## persona_topk_vector (EXPERIMENTAL)

Same personas through the vocabulary bottleneck. Read the logged top-k tokens
(read your data): they show WHAT each persona's mean activation evokes at the
final layer. Steering only makes sense when the two personas' token sets differ;
if both collapse to the same generic sentence-starters, the contrast is ~zero
and the vector is null (a real failure mode of this method on smaller models).
Check that log before trusting any movement in the generations below.

In [ ]:
v_topk = jac.persona_topk_vector(model, tok, optimist, pessimist, k=8)
show_steer(jac, model, tok, v_topk, DEMO, Cs=(-4, 0, 4))

## mean_diff baseline (steering-lite)

The standard activation-difference method on the same prompts and layers, for
comparison. No Jacobian involved: it contrasts mid-layer activations directly.

In [ ]:
from steering_lite import Vector, MeanDiffC

v_md = Vector.train(model, tok, optimist, pessimist, MeanDiffC(layers=tuple(jac.layers)))
show_steer(jac, model, tok, v_md, DEMO, Cs=(-4, 0, 4))

## What to take away

Moving tone at all is not the interesting question. The j-steer-dev specificity
controls asked whether a persona vector moves ITS OWN axis more than an unrelated
persona's vector does, and the persona pullbacks failed that test: they steered
generations, but no more selectively than an unrelated persona's vector did.
Compare the three methods above (persona_vector, persona_topk_vector, mean_diff)
at matched C, but treat all of it as raw material for experiments. If you need
targeted steering, use `word_vector`.